# 🎯 [C204_M03] 음향 스펙트럼 분석 기반 1차 무전 음성 변조 탐지 Baseline 모델 구현
### Google Colab 실습 환경 (지휘통제(C2) / Level 3)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/C204/c204_m03_student.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 지휘통제(C2) (Level 3) |
| **미션 ID** | C204_M03 |
| **미션 주제** | 음향 스펙트럼 분석 기반 1차 무전 음성 변조 탐지 Baseline 모델 구현 |
| **소요 시간** | 약 2.5시간 |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 목표
러시아-우크라이나 전쟁에서는 지휘관 명의의 위조 음성·영상과 무전 기만이 실제로 보고되면서, 생성형 AI 시대에 '들리는 목소리가 정말 그 지휘관인가'를 확인하는 일이 지휘망 보안의 새로운 과제가 되었다. 본 프로젝트에서는 실제 음성 대신 탐지 학습용 합성 신호(배음 신호에 진짜 음성의 자연스러운 흔들림과 문헌상 알려진 위조 흔적을 통계적으로 표시한 추상 신호)를 코드로 만들어, 데이터셋 규격 설계, 무전 채널 잡음 모델링과 MFCC·스펙트로그램 전처리, 규칙 기반 Baseline, 신경망 판별 엔진, 진위 신뢰도 점수, C2 보안 상황판, 통신보안관 2차 검증(HITL), 전자전 재밍 스트레스 시험, AAR 재보정, 최종 통합까지 위조 음성 탐지 파이프라인을 단계별로 구현한다. 음성을 생성하거나 변조하는 공격 기법은 다루지 않으며, 오직 수신 측의 탐지·검증 기능만 구현한다. 최종 목표는 잡음이 심한 무전 환경에서도 위조 명령을 놓치지 않고 오경보를 줄이며, 모든 긴급 명령의 진위 확정이 사람의 인증·청음 확인을 거치도록 설계하는 것이다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

---

## 📂 2. 산출물 디렉토리 체계 및 실습 가이드

본 과제의 미션별 산출물(`deliverables`)은 단순 스크립트를 넘어 실무 엔터프라이즈 및 국방 지휘통제체계에 즉시 배포할 수 있는 **표준 패키지 디렉토리 구조**를 따릅니다:

| 디렉토리 | 주요 산출물(Deliverables) | 역할 및 설명 |
| :--- | :--- | :--- |
| **`data/`** | `data/schema/c204_input_schema.json` | 입출력 데이터 표준 스키마 및 원천/정제 데이터셋 |
| **`src/`** | `src/preprocess/`, `src/models/`, `src/verification/`, `src/ui/`, `src/hitl/`, `src/main_c204.py` | 전처리, Baseline/신경망 모델, 신뢰도 모듈, 대시보드, HITL 등 핵심 파이썬 소스 코드 |
| **`docs/`** | `docs/c204_data_spec.md`, `docs/c204_hitl_guide.md`, `docs/c204_aar_report.md`, `docs/c204_final_report.md` | 데이터 규격서, HITL 가이드, AAR 분석 보고서, 최종 브리핑 등 기술/지휘관 문서 |
| **`configs/`**| `configs/c204_tuned_params.yaml` | AAR 피드백 기반 재조정 임계치 및 환경설정 파일 |
| **`tests/`** | `tests/c204_stress_test.py` | 전자전 재밍/잡음 스트레스 시험 및 단위 테스트 스위트 |

> **💡 [Google Colab / 주피터 노트북 실습 팁]**
> - **방법 A. 자동 파일 저장 (권장):** 코드 셀 첫 줄에 `%%writefile src/preprocess/c204_preprocessor.py`와 같이 매직 명령어를 추가하여 실행하면, 노트북에서 작업한 코드가 해당 폴더의 파일로 자동 추출·저장됩니다.
> - **방법 B. 논리적 모듈 대응:** 별도 파일 추출 없이 노트북 셀 단위로 순차 실행하더라도 각 미션이 해당 모듈의 역할을 1:1로 수행하며, **미션 10(시스템 통합)**에서 전체 파이프라인이 유기적으로 완성되도록 설계되어 있습니다.

---


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


# ----------------------------------------------------------------------------
# [군 표준 산출물 디렉토리 편철 준비 및 가상 데이터셋 무결성 확인]
# ----------------------------------------------------------------------------
output_dirs = [
    'data/schema',
    'docs',
    'src/preprocess',
    'src/models',
    'src/verification',
    'src/ui',
    'src/hitl',
    'tests',
    'configs'
]

print("=== [1] 군 표준 산출물 보관용 폴더 생성 ===")
for d in output_dirs:
    os.makedirs(d, exist_ok=True)
    print(f"  📁 디렉토리 준비 완료: {d}/")

print("\n=== [2] 가상 데이터셋 무결성 점검 ===")
data_dir = 'uploads/C204'
if not os.path.exists(data_dir):
    data_dir = '/home/subokim/Workspace/milicontent2/uploads/C204'

dataset_count = 0
for i in range(1, 11):
    fname = os.path.join(data_dir, f'c204_m{i:02d}_dataset.json')
    if os.path.exists(fname):
        with open(fname, 'r', encoding='utf-8') as f:
            d = json.load(f)
        print(f"  ✅ 데이터셋 확인: c204_m{i:02d}_dataset.json ({d.get('description', '')[:32]}...)")
        dataset_count += 1
    else:
        print(f"  ⚠️ 데이터셋 확인 필요: {fname}")

print(f"\n총 {dataset_count}/10개 가상 데이터셋 무결성 검증 완료. 훈련 준비 완료!\n")


---

## 🎯 미션 03. 음향 스펙트럼 분석 기반 1차 무전 음성 변조 탐지 Baseline 모델 구현
**소요시간**: 약 2.5시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

음향 스펙트럼 분석 기반 1차 무전 음성 변조 및 위조 탐지 Baseline 모델을 구현한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **음향 스펙트럼 단서를 진짜 음성 기준 분포와 비교하는 규칙 기반 1차 위조 탐지 Baseline을 구현한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **고주파 에너지 비율, 프레임 경계 흔적, 배음 대 잡음비, 스펙트럼 평탄도, 저역 에너지, 등록 화자 유사도 6개 단서 산출하고 모델에 적용하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `src/models/c204_baseline_model.py 및 기본 추론 결과 보고서` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 알고리즘의 논리적 흐름 및 군사적 판단 기준의 투명성

#### 미션 PASS 조건

- 정량 기준: 기본 데이터셋 대상 탐지/처리 정확도 75% 이상 달성


- 진행 기준: 단위 기능 동작 확인 및 기본 시나리오 실행 로그 생성


- 검증 방법: 기준 벤치마크 데이터셋 대상 추론 결과 오차율 비교


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
"""음향 스펙트럼 분석으로 구한 위조 단서(고주파 에너지 비율, 프레임 경계 흔적, 배음 대 잡음비, 스펙트럼 평탄도, 저역 에너지, 등록 화자 유사도)를 진짜 음성 기준 분포와 비교하는 규칙 기반 1차 위조 탐지 Baseline을 구현하고, 탐지 정확도·판단 근거·지연시간과 SNR별 한계를 확인한다."""
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import signal
from scipy.fft import dct

# ---------------------------------------------------------------------------
# [교육용가상자료] 탐지 학습용 합성 신호 생성기 (실제 음성·화자·통신망 정보 없음)
# 사람 목소리를 만드는 기능이 아니라, 배음(하모닉) 신호에 '진짜 신호의 자연스러운 흔들림'과
# '알려진 위조 흔적(문헌상 탐지 단서)'의 통계적 특징을 표시해 탐지기를 학습시키기 위한 추상 신호다.
#   bonafide : 자연스러운 피치 흔들림(jitter)·진폭 흔들림(shimmer)·숨소리 잡음
#   tts_art  : 흔들림이 거의 없고 고주파 에너지가 부족하며 20ms 프레임 경계 흔적이 있는 신호
#   vc_art   : 프레임마다 배음 위상이 끊기고 스펙트럼 포락선이 뭉개진 신호
#   replay   : 스피커 재생 대역 제한 + 실내 잔향이 더해진 신호
# ---------------------------------------------------------------------------
FS, DUR = 8000, 0.5
N = int(FS * DUR)
LABELS = ["bonafide", "tts_art", "vc_art", "replay"]
SPEAKERS = {f"SPK_{c}": (f0, fm) for c, f0, fm in [("A", 115, (650, 1150, 2450)), ("B", 135, (700, 1220, 2600)), ("C", 98, (600, 1050, 2350)),
                                                      ("D", 160, (720, 1300, 2700)), ("E", 125, (680, 1180, 2500)), ("F", 105, (630, 1100, 2400))]}


def _source(rng, f0, fm, jitter, shimmer, phase_break=False, hf_cut=None, frame_mod=0.0):
    # [TODO : 학생 실습 3] 음향 스펙트럼 분석 기반 1차 무전 음성 변조 탐지 Baseline 모델 구현 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def make_clip(label, speaker, rng):
    f0, fm = SPEAKERS[speaker]
    if label in ("tts_art", "vc_art"):                               # 위조는 화자 특성을 근사만 함
        f0 = f0 * rng.uniform(0.92, 1.08); fm = tuple(F * rng.uniform(0.95, 1.05) for F in fm)
    if label == "bonafide" or label == "replay":
        x = _source(rng, f0, fm, jitter=rng.uniform(0.008, 0.02), shimmer=rng.uniform(0.05, 0.12))
    elif label == "tts_art":
        x = _source(rng, f0, fm, jitter=rng.uniform(0.0, 0.002), shimmer=0.01, hf_cut=rng.uniform(2300, 2800), frame_mod=rng.uniform(0.08, 0.2))
    else:
        x = _source(rng, f0, fm, jitter=rng.uniform(0.006, 0.015), shimmer=rng.uniform(0.04, 0.1), phase_break=True)
    if label == "replay":                                              # 스피커 대역 + 실내 잔향
        x = signal.sosfilt(signal.butter(4, [450, 2900], "bandpass", fs=FS, output="sos"), x)
        ir = rng.normal(0, 1, int(0.08 * FS)) * np.exp(-np.arange(int(0.08 * FS)) / (0.02 * FS)); ir[0] = 3.0
        x = signal.fftconvolve(x, ir)[:N]
    return x / (np.abs(x).max() + 1e-9)


RADIO_BP = signal.butter(4, [300, 3400], "bandpass", fs=FS, output="sos")


def radio_channel(x, rng, snr_db=15.0, impulse=0.0, dropout=0.0, clip=1.0, jam_tone=0.0):
    """[교육용가상자료] 무전 채널 모형: 300~3400Hz 대역 제한, 배경 잡음(SNR), 충격성 잡음, 신호 끊김, 포화(클리핑), 단일톤 재밍."""
    # [TODO : 학생 실습 3] 음향 스펙트럼 분석 기반 1차 무전 음성 변조 탐지 Baseline 모델 구현 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def make_dataset(n=400, seed=7, snr=(5, 25), labels=LABELS, p=None, claim="SPK_A", **chan):
    """클립 n개: 진짜는 여러 화자, 위조는 모두 지휘관 화자(SPK_A)를 사칭. 반환: (신호 배열, 메타 DataFrame)."""
    rng = np.random.default_rng(seed)
    X, meta = np.zeros((n, N)), []
    if p is None and len(labels) > 1 and labels[0] == "bonafide":        # 기본: 진짜 50%, 나머지 위조 유형 균등
        p = [0.5] + [0.5 / (len(labels) - 1)] * (len(labels) - 1)
    for i in range(n):
        lab = str(rng.choice(labels, p=p))
        spk = str(rng.choice(list(SPEAKERS))) if lab == "bonafide" else claim
        if lab == "replay":
            spk = claim
        s = float(rng.uniform(*snr)) if isinstance(snr, tuple) else float(snr)
        X[i] = radio_channel(make_clip(lab, spk, rng), rng, snr_db=s, **chan)
        meta.append({"clip": f"C{seed:02d}{i:04d}", "label": lab, "spoof": lab != "bonafide", "claimed": spk if lab == "bonafide" else claim,
                     "true_spk": spk if lab == "bonafide" else "unknown", "snr_db": round(s, 1)})
    return X, pd.DataFrame(meta)
# ---------------------------------------------------------------------------

# ---- 전처리·음향 특징(MFCC/스펙트로그램) 공통 함수: 여러 클립을 한 번에 계산 ----
NFFT, HOP = 256, 80


def mel_bank(n_mels=26, fmin=300, fmax=3400):
    mel = lambda f: 2595 * np.log10(1 + f / 700)
    imel = lambda m: 700 * (10 ** (m / 2595) - 1)
    pts = imel(np.linspace(mel(fmin), mel(fmax), n_mels + 2))
    freqs = np.fft.rfftfreq(NFFT, 1 / FS)
    fb = np.zeros((n_mels, len(freqs)))
    for m in range(n_mels):
        l, c, r = pts[m], pts[m + 1], pts[m + 2]
        fb[m] = np.clip(np.minimum((freqs - l) / (c - l), (r - freqs) / (r - c)), 0, None)
    return fb


MEL = mel_bank()
FREQS = np.fft.rfftfreq(NFFT, 1 / FS)


def spectrogram(X, denoise=0.0):
    """배치 전력 스펙트로그램 (클립, 주파수, 프레임). denoise>0이면 잡음 바닥(주파수별 하위 10%)을 그 배수만큼 빼는 스펙트럼 차감."""
    X = np.atleast_2d(X)
    Xp = np.c_[X[:, :1], X[:, 1:] - 0.95 * X[:, :-1]]                                       # 프리엠퍼시스
    frames = np.lib.stride_tricks.sliding_window_view(Xp, NFFT, axis=1)[:, ::HOP] * np.hanning(NFFT)
    P = np.abs(np.fft.rfft(frames, axis=-1)) ** 2
    P = np.transpose(P, (0, 2, 1))
    if denoise:
        floor = np.percentile(P, 10, axis=2, keepdims=True)
        P = np.maximum(P - denoise * floor, 0.05 * P)
    return P


def mfcc(P, n=13):
    logmel = np.log(np.einsum("mf,cft->cmt", MEL, P) + 1e-8)
    return dct(logmel, type=2, axis=1, norm="ortho")[:, :n, :]


def band(P, lo, hi):
    return P[:, (FREQS >= lo) & (FREQS < hi), :].sum(axis=(1, 2))


def jitter_hnr(X):
    """40ms 프레임 자기상관으로 피치 주기를 찾아 피치 흔들림(jitter)과 배음 대 잡음비(HNR 근사)를 구한다."""
    W, lag0, lag1 = 320, int(FS / 250), int(FS / 70)
    fr = np.lib.stride_tricks.sliding_window_view(np.atleast_2d(X), W, axis=1)[:, ::160]
    fr = fr - fr.mean(-1, keepdims=True)
    spec = np.fft.rfft(fr, 2 * W, axis=-1)
    ac = np.fft.irfft(np.abs(spec) ** 2, axis=-1)[..., :W]
    ac = ac / (ac[..., :1] + 1e-12)
    seg = ac[..., lag0:lag1]
    k = seg.argmax(-1)
    peak = seg.max(-1)
    km, kp = np.take_along_axis(seg, np.clip(k - 1, 0, None)[..., None], -1)[..., 0], np.take_along_axis(seg, np.clip(k + 1, None, seg.shape[-1] - 1)[..., None], -1)[..., 0]
    lag = k + lag0 + 0.5 * (km - kp) / (km - 2 * peak + kp - 1e-12)                      # 포물선 보간으로 주기 정밀화
    f0 = FS / lag
    jit = np.median(np.abs(np.diff(f0, axis=1)) / f0[:, 1:], axis=1)
    return jit, np.median(peak, axis=1)


def frame_mod(X):
    """진폭 포락선(2.5ms 간격)의 50Hz 성분 비율: 20ms 프레임 경계 흔적(합성 단서)."""
    env = np.abs(np.atleast_2d(X)).reshape(len(np.atleast_2d(X)), -1, 20).mean(-1)
    sp = np.abs(np.fft.rfft(env - env.mean(1, keepdims=True), axis=1)) ** 2
    f = np.fft.rfftfreq(env.shape[1], 20 / FS)
    return sp[:, (f > 47) & (f < 53)].sum(1) / (sp[:, (f > 30) & (f < 70)].sum(1) + 1e-12)


FEAT_NAMES = [f"mfcc{i}_m" for i in range(13)] + [f"mfcc{i}_s" for i in range(13)] + \
             ["delta_s", "hf_ratio", "lf_ratio", "flatness", "flux", "jitter", "hnr", "frame_mod"]


def extract(X, denoise=0.0):
    """클립 배치 -> 특징 행렬 (클립 x 34) DataFrame."""
    P = spectrogram(X, denoise)
    C = mfcc(P)
    tot = band(P, 300, 3400) + 1e-12
    flat = np.exp(np.log(P[:, (FREQS >= 300) & (FREQS < 3400)] + 1e-10).mean(1)) / (P[:, (FREQS >= 300) & (FREQS < 3400)].mean(1) + 1e-10)
    logP = np.log(P + 1e-8)
    jit, hnr = jitter_hnr(X)
    F = np.c_[C.mean(2), C.std(2), np.abs(np.diff(C, axis=2)).mean((1, 2)), band(P, 2400, 3400) / tot, band(P, 300, 500) / tot,
              flat.mean(1), np.abs(np.diff(logP, axis=2)).mean((1, 2)), jit, hnr, frame_mod(X)]
    return pd.DataFrame(F, columns=FEAT_NAMES)


def enroll(seed=99):
    """지휘관·통신소 화자 6명의 등록 견본(깨끗한 진짜 클립 5개씩의 MFCC 평균)."""
    rng = np.random.default_rng(seed)
    tpl = {}
    for spk in SPEAKERS:
        Xe = np.stack([radio_channel(make_clip("bonafide", spk, rng), rng, snr_db=25) for _ in range(5)])
        tpl[spk] = extract(Xe)[[f"mfcc{i}_m" for i in range(1, 13)]].mean().values
    return tpl


def speaker_sim(F, claimed, tpl):
    """주장 화자 견본과 MFCC 평균 벡터의 코사인 유사도(화자 검증 점수)."""
    v = F[[f"mfcc{i}_m" for i in range(1, 13)]].values
    t = np.stack([tpl[c] for c in claimed])
    return (v * t).sum(1) / (np.linalg.norm(v, axis=1) * np.linalg.norm(t, axis=1) + 1e-12)

# 단서: (특징, 위조일 때 방향, 주로 잡는 위협)
CUES = [("hf_ratio", "low", "tts_art"), ("frame_mod", "high", "tts_art"), ("hnr", "low", "vc_art"),
        ("flatness", "high", "vc_art"), ("lf_ratio", "low", "replay"), ("spk_sim", "low", "tts/vc")]


def fit_rules(F, q=0.03):
    """진짜 음성(bonafide)만으로 단서별 정상 범위 경계를 정한다(한쪽 3% 분위수)."""
    th = {}
    for f, d, _ in CUES:
        th[f] = F[f].quantile(q) if d == "low" else F[f].quantile(1 - q)
    return th


def apply_rules(F, th):
    hits = pd.DataFrame({f: (F[f] < th[f]) if d == "low" else (F[f] > th[f]) for f, d, _ in CUES})
    return hits.any(axis=1), hits


tpl = enroll()
# 기준 설정용: 진짜 음성 200개(여러 화자, SNR 5~25dB)
Xb, Mb = make_dataset(200, seed=7, labels=["bonafide"], p=None)
Fb = extract(Xb); Fb["spk_sim"] = speaker_sim(Fb, Mb["claimed"], tpl)
th = fit_rules(Fb)
print("[규칙 경계(진짜 음성 기준 3% 분위수)]", {k: round(float(v), 3) for k, v in th.items()})

# [교육용가상자료] 평가용 기본 데이터셋: 진짜 50%, 위조 3종 각 약 17%, SNR 5~25dB
X, M = make_dataset(400, seed=8)
t0 = time.perf_counter()
F = extract(X); F["spk_sim"] = speaker_sim(F, M["claimed"], tpl)
pred, hits = apply_rules(F, th)
ms = (time.perf_counter() - t0) * 1000 / len(X)
acc = (pred == M["spoof"]).mean()
print(f"\n클립 {len(X)}개: 탐지 정확도 {acc:.1%} (목표 75% 이상: {'달성' if acc >= 0.75 else '미달'}), "
      f"위조 재현율 {pred[M['spoof']].mean():.1%}, 오경보율(진짜를 위조로) {pred[~M['spoof']].mean():.1%}, 지연 {ms:.2f} ms/클립")

print("\n[유형별 위조 판정 비율과 주로 걸린 단서]")
for lab in LABELS:
    s = M["label"] == lab
    top = hits[s].mean().sort_values(ascending=False)
    print(f"{lab:9s} 위조 판정 {pred[s].mean():6.1%}  주요 단서: {', '.join(f'{k}({v:.0%})' for k, v in top.head(2).items())}")

print("\nSNR 구간   정확도   오경보율")
for lo, hi in [(5, 10), (10, 15), (15, 25)]:
    s = (M["snr_db"] >= lo) & (M["snr_db"] < hi)
    print(f"{lo:>2}~{hi:<2}dB    {(pred[s] == M['spoof'][s]).mean():6.1%}   {pred[s & ~M['spoof']].mean():6.1%}")

ex = M.index[M["spoof"] & pred][0]
print(f"\n판단 근거 예시 {M['clip'][ex]}({M['label'][ex]}): 걸린 단서 {', '.join(hits.columns[hits.loc[ex]])}")
print("한계: 잡음이 큰 채널에서는 진짜 음성의 단서 값도 경계를 넘어 오경보가 늘고, 재생(replay)은 화자 유사도가 높아 단서가 적다 -> 미션 4·5에서 학습형 모델과 SNR 반영 신뢰도로 보완")

plt.figure(figsize=(7, 4))
for lab, c in zip(LABELS, ["tab:green", "tab:red", "tab:purple", "tab:orange"]):
    s = M["label"] == lab
    plt.scatter(F["hf_ratio"][s], F["hnr"][s], s=10, c=c, label=lab)
plt.axvline(th["hf_ratio"], ls="--", c="k"); plt.axhline(th["hnr"], ls="--", c="k")
plt.xlabel("high-frequency energy ratio"); plt.ylabel("HNR (autocorr peak)"); plt.legend(); plt.title("Rule-based cues (synthetic signals)")
plt.tight_layout(); plt.show()


<details>
<summary><b>💡 [참고] 미션 03 예상 실행 결과 (Output)</b></summary>

```text
[규칙 경계(진짜 음성 기준 3% 분위수)] {'hf_ratio': 0.176, 'frame_mod': 0.286, 'hnr': 0.494, 'flatness': 0.341, 'lf_ratio': 0.0, 'spk_sim': 0.702}

클립 400개: 탐지 정확도 78.8% (목표 75% 이상: 달성), 위조 재현율 74.0%, 오경보율(진짜를 위조로) 16.8%, 지연 0.93 ms/클립

[유형별 위조 판정 비율과 주로 걸린 단서]
bonafide  위조 판정  16.8%  주요 단서: frame_mod(9%), hnr(3%)
tts_art   위조 판정 100.0%  주요 단서: frame_mod(100%), hf_ratio(22%)
vc_art    위조 판정  85.9%  주요 단서: hnr(84%), flatness(9%)
replay    위조 판정  30.5%  주요 단서: hnr(14%), spk_sim(12%)

SNR 구간   정확도   오경보율
 5~10dB     77.7%    27.1%
10~15dB     81.7%     7.1%
15~25dB     77.5%    16.1%

판단 근거 예시 C080002(tts_art): 걸린 단서 frame_mod
한계: 잡음이 큰 채널에서는 진짜 음성의 단서 값도 경계를 넘어 오경보가 늘고, 재생(replay)은 화자 유사도가 높아 단서가 적다 -> 미션 4·5에서 학습형 모델과 SNR 반영 신뢰도로 보완
```
</details>
